# Fundamentals of Accelerator Physics and Technology
### Simulations and Measurements Lab
# Computer Lab: Simulated beam transport with quadrupole focusing — Xsuite version
##### Original authors: K. Ruisard, N. Evans and V. S. Morozov  
##### Local Xsuite remake

This notebook replaces the original Sirepo/Elegant workflow with local Python and Xsuite. Questions for students are still numbered.

The front-facing code cells expose the physics knobs that students should be able to change. Plotting, dense line slicing, matching optimization, and particle-distribution generation are kept in `quadrupole_focusing_xsuite_helpers.py`.

<style>
.answer {
    color: #b00020;
    border-left: 4px solid #b00020;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.answer table, .answer th, .answer td {
    color: #b00020;
}
.instructor-note {
    color: #7a3b00;
    border-left: 4px solid #7a3b00;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.note {
    background-color: #f6f8fa;
    padding: 0.75em;
    border-left: 4px solid #999;
}
</style>

<div class="instructor-note">

**Instructor-development note.** This notebook is tagged for a single-source publishing workflow. Keep student-facing prompt, code, widget, and reflection cells tagged `student-keep`. Put worked solutions in separate red cells tagged `solution` and `solution-red`; put solution-only computation cells in cells tagged `solution-code`. The script `make_student_versions.py` removes solution and instructor cells to build the student notebook.

</div>

## 1. Setup

At injection, or at the start of a simulation, there is an optimal spot size and angular divergence for the beam. This is the **matched condition**. In a periodic focusing structure, such as a FODO line, the matched Twiss solution is periodic with the lattice period.

We describe the matched envelope with Twiss parameters $\beta_x$, $\beta_y$, $\alpha_x$, and $\alpha_y$. The RMS beam size is related to beta and geometric emittance by

\[
\sigma_x = \sqrt{\beta_x\epsilon_x}, \qquad \sigma_y = \sqrt{\beta_y\epsilon_y}.
\]

| Parameter | Value |
|---|---:|
| Species | Electron |
| Energy | 1 GeV |
| $\epsilon_x$ | $6\ \mathrm{mm\,mrad}$ |
| $\epsilon_y$ | $6\ \mathrm{mm\,mrad}$ |
| FODO period | $5.0\ \mathrm{m}$ |
| Quadrupole-center spacing | $2.5\ \mathrm{m}$ |
| Quadrupole geometric strength | $K_1 = 0.6\ \mathrm{m}^{-2}$ |
| Quadrupole length | $0.5\ \mathrm{m}$ |

The default FODO sequence starts halfway through a drift between quadrupoles:

`drift — focusing quadrupole — drift — defocusing quadrupole — drift`.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

import quadrupole_focusing_xsuite_helpers as qfh

qfh.set_plotly_renderer("plotly_mimetype")
qfh.print_versions()


xtrack version: 0.107.0
plotly version: 6.5.2
plotly renderer: plotly_mimetype


## 2. Beamline matching

### A) Unmatched beam

Initially, propagate a deliberately unmatched beam through one FODO cell. Use

\[
\beta_x=\beta_y=4\ \mathrm{m}, \qquad \alpha_x=\alpha_y=0.
\]

Run the cell below and compare the beta functions with the RMS beam sizes.


In [2]:
fodo_cell = qfh.make_fodo_line(n_cells=1, k1=0.6)

tw_unmatched_cell = qfh.twiss_dense(
    fodo_cell,
    betx=4.0, alfx=0.0,
    bety=4.0, alfy=0.0,
)

qfh.plot_beta_and_sigma(tw_unmatched_cell, "Unmatched Twiss functions and RMS sizes through one FODO cell")
qfh.twiss_dataframe(tw_unmatched_cell).head(8)


Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

,name,s,betx,bety,alfx,alfy,mux,muy,x,px,y,py
0,D1_c01..0,0.000,4.000000,4.000000,0.00000,0.00000,0.000000,0.000000,0.0,0.0,0.0,0.0
1,D1_c01..1,0.005,4.000006,4.000006,-0.00125,-0.00125,0.000199,0.000199,0.0,0.0,0.0,0.0
2,D1_c01..2,0.010,4.000025,4.000025,-0.00250,-0.00250,0.000398,0.000398,0.0,0.0,0.0,0.0
3,D1_c01..3,0.015,4.000056,4.000056,-0.00375,-0.00375,0.000597,0.000597,0.0,0.0,0.0,0.0
4,D1_c01..4,0.020,4.000100,4.000100,-0.00500,-0.00500,0.000796,0.000796,0.0,0.0,0.0,0.0
5,D1_c01..5,0.025,4.000156,4.000156,-0.00625,-0.00625,0.000995,0.000995,0.0,0.0,0.0,0.0
6,D1_c01..6,0.030,4.000225,4.000225,-0.00750,-0.00750,0.001194,0.001194,0.0,0.0,0.0,0.0
7,D1_c01..7,0.035,4.000306,4.000306,-0.00875,-0.00875,0.001393,0.001393,0.0,0.0,0.0,0.0


### B) Solving for the matched solution

Now ask Xsuite for the periodic one-cell Twiss solution. In the original Sirepo/Elegant lab this was done by setting `Matched = Yes`.

From the matched lattice function, the phase advance is

\[
\psi_x = \int \frac{ds}{\beta_x(s)}, \qquad \psi_y = \int \frac{ds}{\beta_y(s)}.
\]

Xsuite reports `qx` and `qy` as tune-like phase advances in **turns per pass**. For this one-cell line, one pass is one FODO cell. Multiply by $2\pi$ to get radians.

**Q0) Calculate the X and Y phase advances for the single FODO cell.**


In [3]:
tw_cell = qfh.twiss_periodic(fodo_cell)
tw_cell_dense = qfh.twiss_dense(fodo_cell)

qfh.plot_beta_and_sigma(tw_cell_dense, "Matched Twiss functions and RMS sizes for one FODO cell")
qfh.phase_advance_table(tw_cell)


Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

,quantity,value
0,Qx,0.113502
1,Qy,0.113502
2,psi_x [rad],0.713155
3,psi_y [rad],0.713155
4,psi_x [deg],40.860766
5,psi_y [deg],40.860766


<div class="answer">

**Q0 answer.** The one-cell phase advances are equal by symmetry:

\[
Q_x = Q_y = 0.113502\ \text{turns/cell}.
\]

Equivalently,

\[
\psi_x=\psi_y=2\pi Q = 0.713155\ \mathrm{rad}=40.861^\circ.
\]

</div>


### Try it: change the FODO strength

Move the sliders and watch how the beta functions, beam size, and phase advance change. This reproduces the useful interactivity of the old Sirepo version while keeping the Xsuite workflow visible.


In [ ]:
qfh.interactive_fodo_strength()


Interactive widget cell not pre-executed in this distributed solutions notebook. Run this cell in Jupyter to activate the sliders.


The thin-lens approximation gives the following FODO estimates:

\[
\beta_{\max}=L \frac{1 + \sin( \psi/2)}{\sin \psi},
\qquad
\beta_{\min}=L \frac{1 - \sin(\psi/2)}{\sin \psi}.
\]

<div class="note">
The original worksheet states that the half-cell length is $2.5\ \mathrm{m}$, but the thin-lens comparison described as “quite close” is obtained by using the full $5.0\ \mathrm{m}$ FODO period in the formula above. The code below makes that convention explicit as `length_for_formula=5.0`.
</div>

**Q1) For this cell, calculate $\beta_{\min}$ and $\beta_{\max}$ in two ways:**

- A) thin-lens prediction
- B) using Xsuite

**Q2) If you increased the length of the quadrupole elements while holding both the cell length and the phase advance $\psi$ fixed, would the difference between the thin-lens and Xsuite results get larger or smaller? Explain.**


In [4]:
qfh.thin_lens_comparison_table(tw_cell_dense, length_for_formula=5.0)


,quantity,value
0,beta_min thin lens [m],4.974840
1,beta_max thin lens [m],10.310466
2,beta_min Xsuite dense [m],5.031928
3,beta_max Xsuite dense [m],10.189479
4,thin-lens min error [%],-1.134519
5,thin-lens max error [%],1.187376


<div class="answer">

**Q1 answer.** With $L=5.0\ \mathrm{m}$ in the printed thin-lens formula,

\[
\beta_{\min}^{\mathrm{thin}}=4.97484\ \mathrm{m},\qquad
\beta_{\max}^{\mathrm{thin}}=10.31047\ \mathrm{m}.
\]

Dense Xsuite sampling of the thick-lens cell gives

\[
\beta_{\min}^{\mathrm{Xsuite}}=5.03193\ \mathrm{m},\qquad
\beta_{\max}^{\mathrm{Xsuite}}=10.18948\ \mathrm{m}.
\]

The values are close, but not identical, because the Xsuite model uses finite-length quadrupoles rather than point lenses.

</div>


In [5]:
quad_length_scan = qfh.quad_length_fixed_phase_table(target_q=tw_cell.qx)
quad_length_scan


Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

,quad length [m],|k1| for same phase [1/m^2],beta_min thick [m],beta_max thick [m],|thick-thin| beta_min [m],|thick-thin| beta_max [m]
0,0.1,2.830560,4.986389,10.286419,0.011549,0.024047
1,0.2,1.435094,4.997879,10.262277,0.023039,0.048189
2,0.5,0.600000,5.031928,10.189479,0.057088,0.120987
3,0.8,0.393669,5.065154,10.116664,0.090315,0.193803
4,1.1,0.302145,5.097198,10.044814,0.122358,0.265652
5,1.4,0.252168,5.127475,9.975460,0.152636,0.335006
6,1.8,0.215402,5.163359,9.891329,0.188519,0.419137
7,2.2,0.197754,5.189878,9.827511,0.215039,0.482955


<div class="answer">

**Q2 answer.** The difference gets larger as the quadrupoles become longer, even when $\psi$ is held fixed by adjusting $k_1$. The thin-lens approximation assumes focusing is localized at zero-length lenses. Longer quadrupoles make the focusing more distributed, so the thick-lens model deviates more from the thin-lens formula.

The scan above tests this directly. For example, in this model the $\beta_{\max}$ difference grows from about $0.024\ \mathrm{m}$ at a $0.10\ \mathrm{m}$ quadrupole length to about $0.483\ \mathrm{m}$ at a $2.20\ \mathrm{m}$ quadrupole length.

</div>


### Try it: test the thick-lens effect directly

This slider changes the quadrupole length. The helper then adjusts $k_1$ so that the one-cell phase advance remains the same as the default cell. Watch the thick-lens beta extrema move away from the fixed thin-lens prediction.


In [ ]:
qfh.interactive_quad_length_effect(target_q=tw_cell.qx)


Interactive widget cell not pre-executed in this distributed solutions notebook. Run this cell in Jupyter to activate the sliders.


**Q3) What are the average, maximum, and minimum RMS beam spot sizes for a matched beam in this lattice?**

Use $\sigma=\sqrt{\beta\epsilon}$ with $\epsilon_x=\epsilon_y=6\ \mathrm{mm\,mrad}$.


In [6]:
qfh.beam_size_summary(tw_cell_dense, label="matched FODO cell")


,quantity,matched FODO cell
0,length [m],5.000000
1,Qx,0.113502
2,Qy,0.113502
3,phase x [deg],40.860766
4,phase y [deg],40.860766
5,min beta_x [m],5.031928
6,max beta_x [m],10.189479
7,min beta_y [m],5.031928
8,max beta_y [m],10.189479
9,mean sigma_x [mm],6.607639


<div class="answer">

**Q3 answer.** For the matched cell and equal geometric emittances of $6\ \mathrm{mm\,mrad}$,

| Quantity | Value |
|---|---:|
| $\langle\sigma_x\rangle_s$ | $6.608\ \mathrm{mm}$ |
| $\langle\sigma_y\rangle_s$ | $6.608\ \mathrm{mm}$ |
| $\max \sigma_x$ | $7.819\ \mathrm{mm}$ |
| $\max \sigma_y$ | $7.819\ \mathrm{mm}$ |
| $\min \sigma_x$ | $5.495\ \mathrm{mm}$ |
| $\min \sigma_y$ | $5.495\ \mathrm{mm}$ |
| $\max(\sigma_x/\sigma_y)$ | $1.423$ |

</div>


**Q4) Where along the lattice is the beam round? Where is it largest and smallest in the horizontal plane? In the vertical plane? Give your answer in $s$ and in terms of elements or regions.**


In [7]:
qfh.location_table(tw_cell_dense)


,condition,s [m],lattice region,sigma_x [mm],sigma_y [mm]
0,round beam,"0.00, 2.50, 5.00",mid-drifts / symmetry points,same as sigma_y,same as sigma_x
1,max sigma_x,1.25,focusing quadrupole QF,7.819007,5.494685
2,min sigma_x,3.75,defocusing quadrupole QD,5.494685,7.819007
3,max sigma_y,3.75,defocusing quadrupole QD,5.494685,7.819007
4,min sigma_y,1.25,focusing quadrupole QF,7.819007,5.494685


<div class="answer">

**Q4 answer.** The beam is round at the symmetry points $s=0.00\ \mathrm{m}$, $s=2.50\ \mathrm{m}$, and $s=5.00\ \mathrm{m}$. The horizontal beam size is largest at $s\approx1.25\ \mathrm{m}$, inside the focusing quadrupole QF, and smallest at $s\approx3.75\ \mathrm{m}$, inside the defocusing quadrupole QD. The vertical plane does the opposite: $\sigma_y$ is largest at $s\approx3.75\ \mathrm{m}$ in QD and smallest at $s\approx1.25\ \mathrm{m}$ in QF.

</div>


### C) Matched beam propagation down a 100 m FODO beamline

Now repeat the same FODO cell 20 times. The total line length is $20\times5\ \mathrm{m}=100\ \mathrm{m}$.

**Q5) Confirm that the tune of this 20-cell lattice is consistent with the one-cell solution.**


In [8]:
fodo_100m = qfh.make_fodo_line(n_cells=20, k1=0.6)
tw_100m = qfh.twiss_periodic(fodo_100m)
tw_100m_dense = qfh.twiss_dense(fodo_100m, points_per_meter=30)

qfh.plot_sigmas(tw_100m_dense, "Matched RMS beam sizes through 20 FODO cells")

pd.DataFrame({
    "quantity": ["nu_x over 100 m", "nu_y over 100 m", "psi_x over 100 m [rad]", "psi_y over 100 m [rad]", "nu_x per cell", "nu_y per cell", "psi_x per cell [rad]", "psi_y per cell [rad]"],
    "value": [tw_100m.qx, tw_100m.qy, 2*np.pi*tw_100m.qx, 2*np.pi*tw_100m.qy, tw_100m.qx/20, tw_100m.qy/20, 2*np.pi*tw_100m.qx/20, 2*np.pi*tw_100m.qy/20],
})


Slicing line:   0%|          | 0/100 [00:00<?, ?it/s]

,quantity,value
0,nu_x over 100 m,2.270043
1,nu_y over 100 m,2.270043
2,psi_x over 100 m [rad],14.263098
3,psi_y over 100 m [rad],14.263098
4,nu_x per cell,0.113502
5,nu_y per cell,0.113502
6,psi_x per cell [rad],0.713155
7,psi_y per cell [rad],0.713155


<div class="answer">

**Q5 answer.** Over 100 m,

\[
\nu_x=\nu_y=2.27004,
\qquad
\psi_x=\psi_y=14.2631\ \mathrm{rad}=817.215^\circ.
\]

Dividing by 20 cells gives

\[
\nu_x/20=\nu_y/20=0.113502,
\qquad
\psi_x/20=\psi_y/20=0.713155\ \mathrm{rad}=40.861^\circ,
\]

which is the same as the one-cell result.

</div>


**Q6) How many oscillations does a single particle make in 100 m?**

You can calculate this from the tune, but the cell below also launches a 1 mm horizontal centroid offset. The centroid follows the same linear equations as a single particle.


In [9]:
tw_centroid = qfh.centroid_orbit(fodo_100m, x0=1e-3, px0=0.0)
qfh.plot_centroid(tw_centroid)


<div class="answer">

**Q6 answer.** A single particle makes about

\[
\nu_x=2.270
\]

horizontal betatron oscillations over the 100 m line. The same value holds vertically for this symmetric FODO cell.

</div>


### D) Propagation of a mismatched beam

Initialize the beam with a 10% beta mismatch relative to the matched solution:

\[
\beta_x = 1.1\beta_{x,\mathrm{matched}},\qquad
\beta_y = 1.1\beta_{y,\mathrm{matched}},
\]

while keeping the matched alpha values.

**Q7) Count the approximate number of envelope mismatch oscillations over the 100 m beamline. Does it agree with the smooth-approximation prediction $\nu_{\mathrm{envelope}}=2\nu$?**


In [10]:
tw_mismatch_dense = qfh.twiss_dense(
    fodo_100m,
    points_per_meter=30,
    betx=float(tw_cell.betx[0]) * 1.10,
    alfx=float(tw_cell.alfx[0]),
    bety=float(tw_cell.bety[0]) * 1.10,
    alfy=float(tw_cell.alfy[0]),
)

qfh.plot_mismatch(tw_100m_dense, tw_mismatch_dense, "Matched and 10% mismatched RMS beam envelopes")
qfh.plot_cell_boundary_envelope(tw_mismatch_dense)
qfh.cell_start_envelope_table(tw_mismatch_dense).head(10)


Slicing line:   0%|          | 0/100 [00:00<?, ?it/s]

,cell index,s [m],sigma_x [mm],sigma_y [mm]
0,0,0.0,6.896328,6.896328
1,1,5.0,6.272609,7.010637
2,2,10.0,6.210476,6.433746
3,3,15.0,6.823517,6.129432
4,4,20.0,7.045350,6.650182
5,5,25.0,6.521699,7.068557
6,6,30.0,6.116382,6.705396
7,7,35.0,6.560445,6.146730
8,8,40.0,7.055702,6.380974
9,9,45.0,6.789369,6.981542


<div class="answer">

**Q7 answer.** The single-particle tune over 100 m is $\nu=2.270$, so the smooth-approximation envelope prediction is

\[
\nu_{\mathrm{envelope}}=2\nu=4.540.
\]

The simulated mismatch envelope shows about $4.5$ slow envelope oscillations over 100 m, so it agrees well with the prediction. The important point is to count the slow modulation of the envelope, not every local FODO-cell maximum of $\sigma_x(s)$.

</div>


### Try it: change the injection mismatch

The slider changes the initial beta mismatch factor. Values closer to 1 are closer to matched; values farther from 1 produce stronger beating.


In [ ]:
qfh.interactive_mismatch(tw_cell)


Interactive widget cell not pre-executed in this distributed solutions notebook. Run this cell in Jupyter to activate the sliders.


### E) Matched solution with decreased quadrupole strength

Now reduce the quadrupole strength while keeping the emittance fixed. The RMS envelope equation contains a focusing term and an emittance-pressure term,

\[
\frac{d^2 \sigma_x}{ds^2}=-K_x\sigma_x+\frac{\epsilon_x^2}{\sigma_x^3}.
\]

With weaker quadrupoles, the focusing term is weaker relative to the emittance term.

**Q8) For a new FODO cell with $|k_1|=0.2\ \mathrm{m}^{-2}$, record the matched beam-size statistics and phase advance.**


In [11]:
fodo_cell_weak = qfh.make_fodo_line(n_cells=1, k1=0.2)
tw_cell_weak = qfh.twiss_periodic(fodo_cell_weak)
tw_cell_weak_dense = qfh.twiss_dense(fodo_cell_weak)

qfh.plot_beta_and_sigma(tw_cell_weak_dense, "Matched weak FODO cell: |k1| = 0.2 1/m^2")
display(qfh.beam_size_summary(tw_cell_weak_dense, label="weak FODO cell"))
display(qfh.phase_advance_table(tw_cell_weak))


Slicing line:   0%|          | 0/5 [00:00<?, ?it/s]

,quantity,weak FODO cell
0,length [m],5.000000
1,Qx,0.037125
2,Qy,0.037125
3,phase x [deg],13.364990
4,phase y [deg],13.364990
5,min beta_x [m],19.187220
6,max beta_x [m],24.053987
7,min beta_y [m],19.187220
8,max beta_y [m],24.053987
9,mean sigma_x [mm],11.362211


,quantity,value
0,Qx,0.037125
1,Qy,0.037125
2,psi_x [rad],0.233263
3,psi_y [rad],0.233263
4,psi_x [deg],13.364990
5,psi_y [deg],13.364990


<div class="answer">

**Q8 answer.** For $|k_1|=0.2\ \mathrm{m}^{-2}$,

| Quantity | Value |
|---|---:|
| $\langle\sigma_x\rangle_s$ | $11.362\ \mathrm{mm}$ |
| $\langle\sigma_y\rangle_s$ | $11.362\ \mathrm{mm}$ |
| $\max\sigma_x$ | $12.013\ \mathrm{mm}$ |
| $\max\sigma_y$ | $12.013\ \mathrm{mm}$ |
| $\min\sigma_x$ | $10.730\ \mathrm{mm}$ |
| $\min\sigma_y$ | $10.730\ \mathrm{mm}$ |
| $\max(\sigma_x/\sigma_y)$ | $1.120$ |
| $Q_x=Q_y$ | $0.037125$ turns/cell |
| $\psi_x=\psi_y$ | $0.233263\ \mathrm{rad}=13.365^\circ$ |

The average beam size is larger than in the stronger cell, and the aspect ratio variation is smaller.

</div>


### Try it: vary the matched cell strength

Use this slider to connect the strong-cell and weak-cell limits continuously.


In [ ]:
qfh.interactive_weak_cell()


Interactive widget cell not pre-executed in this distributed solutions notebook. Run this cell in Jupyter to activate the sliders.


### F) Injection mismatch between different FODO sections

Build a hybrid beamline containing 10 cells with $|k_1|=0.6\ \mathrm{m}^{-2}$ followed by 10 cells with $|k_1|=0.5\ \mathrm{m}^{-2}$. Start the line with the matched Twiss parameters of the strong cell.

**Q9) Describe what happens as the beam transitions from the stronger to the weaker FODO cell. Is the beam matched in the first 10 cells? What about the last 10 cells?**


In [12]:
hybrid_line = qfh.make_hybrid_fodo_line(first_cells=10, second_cells=10, k1_first=0.6, k1_second=0.5)

tw_hybrid_from_strong_match = qfh.twiss_dense(
    hybrid_line,
    points_per_meter=30,
    betx=float(tw_cell.betx[0]), alfx=float(tw_cell.alfx[0]),
    bety=float(tw_cell.bety[0]), alfy=float(tw_cell.alfy[0]),
)

qfh.plot_hybrid_transition(tw_hybrid_from_strong_match, transition_s=50.0)
qfh.optics_summary(tw_hybrid_from_strong_match, label="strong-cell initial match")


Slicing line:   0%|          | 0/100 [00:00<?, ?it/s]

,quantity,strong-cell initial match
0,length [m],100.000000
1,Qx,NaN
2,Qy,NaN
3,phase x [deg],NaN
4,phase y [deg],NaN
5,min beta_x [m],4.819304
6,max beta_x [m],15.308358
7,min beta_y [m],4.928548
8,max beta_y [m],15.288913
9,mean sigma_x [mm],7.001715


<div class="answer">

**Q9 answer.** The beam is matched in the first 10 cells because its initial Twiss parameters are the periodic solution of the $|k_1|=0.6\ \mathrm{m}^{-2}$ FODO cell. At $s=50\ \mathrm{m}$ the lattice changes to weaker $|k_1|=0.5\ \mathrm{m}^{-2}$ cells. The strong-cell Twiss parameters are not the periodic solution of the weaker cell, so the beam becomes mismatched and the envelope starts beating in the last 10 cells.

</div>


### Try it: change the downstream focusing strength

The first section remains fixed. Change the second-section strength and observe the mismatch at the transition.


In [ ]:
qfh.interactive_hybrid_transition(tw_cell)


Interactive widget cell not pre-executed in this distributed solutions notebook. Run this cell in Jupyter to activate the sliders.


Now ask Xsuite for the periodic solution of the entire 100 m hybrid line.

**Q10) Is this new solution matched over the period $L=100\ \mathrm{m}$? Is it matched for the period $L=5\ \mathrm{m}$? Does there exist one solution that is matched for period $L=5\ \mathrm{m}$ in all 20 cells?**


In [13]:
tw_hybrid_periodic = qfh.twiss_periodic(hybrid_line)
tw_hybrid_periodic_dense = qfh.twiss_dense(hybrid_line, points_per_meter=30)

qfh.plot_hybrid_transition(tw_hybrid_periodic_dense, transition_s=50.0, title="Periodic solution of the full 100 m hybrid line")

pd.DataFrame({
    "quantity": ["beta_x start [m]", "beta_x end [m]", "alpha_x start", "alpha_x end", "Qx over 100 m", "Qy over 100 m"],
    "value": [tw_hybrid_periodic.betx[0], tw_hybrid_periodic.betx[-1], tw_hybrid_periodic.alfx[0], tw_hybrid_periodic.alfx[-1], tw_hybrid_periodic.qx, tw_hybrid_periodic.qy],
})


Slicing line:   0%|          | 0/100 [00:00<?, ?it/s]

,quantity,value
0,beta_x start [m],5.627751
1,beta_x end [m],5.627751
2,alpha_x start,-0.936788
3,alpha_x end,-0.936788
4,Qx over 100 m,2.070325
5,Qy over 100 m,2.070325


<div class="answer">

**Q10 answer.** The Xsuite periodic solution is matched over the full 100 m hybrid period. For example,

\[
\beta_x(0)=\beta_x(100\ \mathrm{m})=5.62775\ \mathrm{m},
\qquad
\alpha_x(0)=\alpha_x(100\ \mathrm{m})=-0.93679.
\]

It is not matched with a 5 m period, because the lattice is not the same in every 5 m cell: the first 10 cells and last 10 cells have different quadrupole strengths. There is no single Twiss solution that is matched every 5 m in all 20 cells. Each cell type has its own local 5 m matched solution; a transition between them requires a matching section or accepts mismatch.

</div>


### G) Matching-section insert

A real accelerator contains insertions, injection and extraction regions, experiments, and other sections that are not just repeated FODO cells. A **matching section** uses adjustable quadrupoles to transform the incoming Twiss parameters into the desired outgoing Twiss parameters.

Here the incoming beam is taken from the end of the matched FODO cell. The target at the end of the matching section is a round, uncorrelated distribution:

\[
\beta_x=\beta_y, \qquad \alpha_x=0, \qquad \alpha_y=0.
\]

The section has four quadrupoles. Xsuite varies their strengths to satisfy the endpoint constraints.


In [14]:
initial_from_fodo_end = qfh.fodo_end_twiss_dict(tw_cell)

# Keep a copy of the unoptimized section for the before/after plot.
match_section_before = qfh.make_match_section()
tw_match_before_dense = qfh.twiss_dense(match_section_before, points_per_meter=80, **initial_from_fodo_end)

# Optimize a fresh section.
match_section = qfh.make_match_section()
opt, tw_match_before, tw_match_after = qfh.match_round_section(match_section, initial_from_fodo_end)
tw_match_after_dense = qfh.twiss_dense(match_section, points_per_meter=80, **initial_from_fodo_end)

qfh.plot_matching_comparison(tw_match_before_dense, tw_match_after_dense)
qfh.matching_summary(match_section, tw_match_after)


Slicing line:   0%|          | 0/9 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/9 [00:00<?, ?it/s]

,quantity,value
0,final beta_x [m],7.426127e+00
1,final beta_y [m],7.426127e+00
2,final beta_x - beta_y [m],-3.190246e-09
3,final alpha_x,2.329526e-09
4,final alpha_y,1.379939e-10
5,QM1 k1 [1/m^2],4.277798e-01
6,QM2 k1 [1/m^2],-6.671136e-01
7,QM3 k1 [1/m^2],7.367798e-01
8,QM4 k1 [1/m^2],-5.142089e-01


### Try it: match by hand

Before relying on the optimizer, adjust the four quadrupole knobs manually. The endpoint target is $\beta_x-\beta_y\to0$, $\alpha_x\to0$, and $\alpha_y\to0$.


In [ ]:
qfh.interactive_manual_match(initial_from_fodo_end)


Interactive widget cell not pre-executed in this distributed solutions notebook. Run this cell in Jupyter to activate the sliders.


**Q11) Attach graphs of the input and output $xx'$ distributions.**

**Q12) Attach graphs of $\beta_{x,y}$ before and after optimization.**

**Q13) What are the final values of $\beta_{x,y}$ at the end of `MATCHsection`?**

**Q14) What are the optimized strengths of the quadrupoles in `MATCHsection`?**

**Q15) Extra credit: design an entire injection insertion and show the resulting optics.**


In [15]:
tracked_distribution = qfh.track_distribution(match_section, initial_from_fodo_end, n_particles=2500, seed=11)
qfh.plot_tracked_xpx_distribution(tracked_distribution, "Tracked input/output horizontal phase-space distributions")
qfh.plot_phase_space_ellipses(tw_match_before, tw_match_after)


In [16]:
insertion_line = qfh.make_injection_insertion_line(fodo_cell, match_section)
tw_insertion_dense = qfh.twiss_dense(insertion_line, points_per_meter=60, **initial_from_fodo_end)
qfh.plot_twiss(tw_insertion_dense, "Example FODO + matching section + reversed matching section insertion")


Slicing line:   0%|          | 0/23 [00:00<?, ?it/s]

<div class="answer">

**Q11 answer.** The tracked input/output $xx'$ distribution plots are generated by the preceding code cell. The output distribution is transformed by the optimized matching section; the accompanying ellipse plot shows the corresponding input/output one-rms Twiss ellipses.

**Q12 answer.** The before/after $\beta_{x,y}$ graph is generated in the matching-section optimization cell. The optimized optics satisfy the endpoint constraints while using the four quadrupoles as matching knobs.

**Q13 answer.** At the end of `MATCHsection`,

\[
\beta_x=7.42613\ \mathrm{m},\qquad \beta_y=7.42613\ \mathrm{m},
\]

with

\[
\alpha_x\approx2.33\times10^{-9},\qquad \alpha_y\approx1.38\times10^{-10}.
\]

The alphas are zero to numerical precision.

**Q14 answer.** The optimized quadrupole strengths are

| Quadrupole | $k_1$ [$\mathrm{m}^{-2}$] |
|---|---:|
| QM1 | $+0.42778$ |
| QM2 | $-0.66711$ |
| QM3 | $+0.73678$ |
| QM4 | $-0.51421$ |

**Q15 answer.** The optional insertion plot is generated by the final code cell. It constructs a simple `FODO + MATCHsection + reversed MATCHsection` insertion and plots the resulting beta functions.

</div>
